In [1]:
%cd /content/drive/MyDrive/github/Breakout

/content/drive/MyDrive/github/Breakout


In [2]:
from pathlib import Path
import torch
import torch.nn as nn
import random
from torch.optim import AdamW
from src.models import *
from src.buffer import *
from collections import deque
import numpy as np
import gymnasium as gym
import ale_py
import time

gym.register_envs(ale_py)
print(f'Device: {device}')
ROOT = Path.cwd()
saved_path = ROOT / 'checkpoint' / 'checkpoint.tar'
env = gym.make('ALE/Breakout-v5', obs_type='ram', mode=0, difficulty=0, repeat_action_probability=0)



Device: cpu


In [3]:
if __name__ == "__main__":

    q = QNetwork().to(device)
    q_target = QNetwork().to(device)
    optimizer = AdamW(q.parameters())
    buffer = ReplayBuffer()

    epsilon = 1.0
    best_reward = 0

    try:
        if device == 'cpu':
            checkpoint = torch.load(saved_path, weights_only=False, map_location='cpu')
        else:
            checkpoint = torch.load(saved_path, weights_only=False)
        q.load_state_dict(checkpoint['model'])
        q_target.load_state_dict(checkpoint['model'])
        optimizer.load_state_dict(checkpoint['optimizer'])
        buffer.buffer = checkpoint['buffer']
        epsilon = checkpoint['epsilon']
        best_reward = checkpoint['best_reward']
        print('Checkpoint Loaded')
    except Exception as e:
        print(e)
        print('Checkpoint Not Loaded')

    # Parameters
    gamma = 0.99
    batch_size = 64

    epsilon_max = 1.0
    epsilon_min = 0.05
    random_episodes = 10000

    target_update = 100
    print_episode = 100
    reward_history = deque(maxlen=print_episode)

    step = 0
    episode = 0
    now = time.time()

    while True:
        episode += 1
        s, _ = env.reset(seed=42)
        done = False
        episode_reward = 0

        # Start One Episode
        while not done:
            if random.random() < epsilon:
                a = env.action_space.sample()
            else:
                with torch.no_grad():
                    qs = q(torch.tensor(s, dtype=torch.float32, device=device))
                    a = qs.argmax().item()

            s2, r, terminated, truncated, _ = env.step(a)
            done = terminated or truncated

            buffer.push(s, a, r, s2, done)

            s = s2
            episode_reward += r
            step += 1

            if len(buffer) > batch_size:
                states, actions, rewards, next_states, dones = buffer.sample(batch_size)

                # q(states) : (B, ACTION_SIZE)
                # actions : (B, )

                q_values = q(states).gather(1, actions.unsqueeze(1)).squeeze()  # (B, )

                with torch.no_grad():
                    next_q = q_target(next_states).max(1)[0]
                    target = rewards + gamma * next_q * (1 - dones)

                loss = nn.functional.huber_loss(q_values, target)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            if step % target_update == 0:
                q_target.load_state_dict(q.state_dict())

        reward_history.append(int(episode_reward))
        mean_reward = np.mean(reward_history)

        epsilon = max(epsilon - (epsilon_max - epsilon_min) / random_episodes, epsilon_min)

        if mean_reward >= best_reward:
            best_reward = episode_reward

        if mean_reward >= 40:
            break

        if episode % print_episode == 0:

            torch.save({
                'model': q.state_dict(),
                'optimizer': optimizer.state_dict(),
                'buffer': buffer.buffer,
                'epsilon': epsilon,
                'best_reward': best_reward
            }, saved_path)

            print(f"Episode: {episode} |",
                  f"reward: {mean_reward:.2f} |",
                  f"Best Score: {best_reward:.2f} |",
                  f"Epsilon: {epsilon:.2f} |",
                  f"time: {int(time.time() - now)}s"
                )

            now = time.time()



Checkpoint Loaded
Episode: 100 | reward: 4.34 | Best Score: 11.00 | Epsilon: 0.14 | time: 81s
Episode: 200 | reward: 4.40 | Best Score: 11.00 | Epsilon: 0.13 | time: 79s
Episode: 300 | reward: 5.28 | Best Score: 11.00 | Epsilon: 0.12 | time: 83s
Episode: 400 | reward: 5.05 | Best Score: 11.00 | Epsilon: 0.11 | time: 96s
Episode: 500 | reward: 4.42 | Best Score: 11.00 | Epsilon: 0.10 | time: 95s
Episode: 600 | reward: 4.31 | Best Score: 11.00 | Epsilon: 0.09 | time: 91s
Episode: 700 | reward: 4.90 | Best Score: 11.00 | Epsilon: 0.08 | time: 93s
Episode: 800 | reward: 5.25 | Best Score: 11.00 | Epsilon: 0.07 | time: 99s
Episode: 900 | reward: 5.01 | Best Score: 11.00 | Epsilon: 0.06 | time: 96s
Episode: 1000 | reward: 3.78 | Best Score: 11.00 | Epsilon: 0.05 | time: 91s
Episode: 1100 | reward: 4.75 | Best Score: 11.00 | Epsilon: 0.05 | time: 97s
Episode: 1200 | reward: 4.59 | Best Score: 11.00 | Epsilon: 0.05 | time: 98s
Episode: 1300 | reward: 4.80 | Best Score: 11.00 | Epsilon: 0.05 | 

KeyboardInterrupt: 